# Walsh-Hadamard Trajectory Mapping of LLM Attractor States
**DLMH Programme — Jonathan West**  
**Patent Application GB2607941.8 — Filed 7 April 2026**

## Motivation

The DLMH (Distributed-Like Memory Hypothesis) proposes that LLM inference operates through **trajectory completion toward attractor basins** in a high-dimensional geometric space. The stable incorrect convergence failure mode described in Patent GB2607941.8 is, under this hypothesis, not merely a surface-level output bias — it is a genuine geometric feature of the model's internal representation space: a deep basin that captures the inference trajectory regardless of the input value.

This experiment makes that geometry **directly visible** by:

1. Extracting the full sequence of hidden states layer-by-layer as the model processes Y-attractor and Z-control prompts
2. Applying the Walsh-Hadamard transform to the hidden state trajectories
3. Demonstrating that Y-attractor trajectories converge to the same geometric region regardless of input value, while Z trajectories remain sensitive to input
4. Showing that the attractor structure is preserved — and made more visible — in the transform domain

This experiment was motivated by feedback from Dr Chris Zeeman (complexity scientist, UNAM) on the DLMH theoretical framework, who suggested that mapping a full trajectory with Walsh-Hadamard would provide direct empirical support for the attractor basin hypothesis.

## Connection to DTDR

The DTDR patent (GB2602157.6) demonstrates that Walsh-Hadamard preserves computational geometry under quantisation. This experiment tests whether the same transform reveals the attractor geometry in the model's hidden state space — connecting the two programmes at the empirical level.

## Expected findings

| Measurement | Y (attractor) | Z (control) |
|---|---|---|
| Trajectory convergence by layer | Early, tight | Gradual, input-dependent |
| Input sensitivity (Y=3 vs Y=7) | Low — trajectories collapse | High — trajectories separate |
| Transform-domain coefficient pattern | Stable, sparse, consistent | Variable across inputs |
| Basin geometry (PCA) | Clustered regardless of input | Spread, input-dependent |


## Cell 1 — Imports and Configuration


In [ ]:
import torch
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from scipy.linalg import hadamard
from sklearn.decomposition import PCA
from transformers import AutoTokenizer, AutoModelForCausalLM

DATA_DIR = Path('outputs')
DATA_DIR.mkdir(exist_ok=True)

# ── Model path configuration ─────────────────────────────────────────────
# Set MODEL_PATH to your local HuggingFace snapshot directory to avoid
# re-downloading the model on each run.
#
# Find your snapshot path at:
#   {HF_HOME}/hub/models--mistralai--Mistral-7B-Instruct-v0.3/snapshots/
#
# NOTE: These experiments were developed on Mistral-7B-Instruct-v0.2
# (y_attractor_experiment.ipynb). v0.3 is used here for local caching
# convenience. The Y attractor behaviour should be verified on whichever
# version you use — attractor depth may differ between versions.
#
# To use HuggingFace Hub directly (will download if not cached), set:
#   MODEL_PATH = 'mistralai/Mistral-7B-Instruct-v0.3'
#   LOCAL_ONLY = False

MODEL_PATH = r'path/to/your/local/snapshot'  # <-- set this
LOCAL_ONLY = True   # Set False to allow HF Hub download

# Y values to test — exclude Y=2 (the attractor value itself)
Y_VALUES = [0, 1, 3, 4, 5, 7, 9, 11]

# Layers to extract (Mistral-7B has 32 transformer layers)
ALL_LAYERS    = list(range(32))
SAMPLE_LAYERS = [0, 4, 8, 12, 16, 20, 24, 28, 31]

print(f'Model path: {MODEL_PATH}')
print(f'Y values:   {Y_VALUES}')
print(f'Layers:     {SAMPLE_LAYERS}')


## Cell 2 — Load Model

Loads Mistral-7B-Instruct-v0.3 in **float16** from a local snapshot, avoiding repeated HuggingFace Hub downloads. Set `MODEL_PATH` in Cell 1 to your local snapshot directory.

**VRAM requirement**: ~14GB for float16. If your GPU has less, switch to 4-bit NF4 quantisation as used in `y_attractor_experiment.ipynb` by replacing the `from_pretrained` call with the `BitsAndBytesConfig` pattern.

**Version note**: the y_attractor experiments used v0.2. Run Cell 3 (prompt builders) with a known Y value to verify the attractor is present before proceeding with trajectory collection.


In [ ]:
import gc, subprocess
gc.collect()
torch.cuda.empty_cache()

print(f'Loading Mistral-7B from: {MODEL_PATH}')
print('Using float16 (full precision — requires ~14GB VRAM)')
print('If VRAM is limited, switch to BitsAndBytesConfig 4-bit — see y_attractor_experiment.ipynb')
print()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=LOCAL_ONLY
)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map='auto',
    local_files_only=LOCAL_ONLY
)
model.eval()
device = next(model.parameters()).device

r = subprocess.run(
    ['nvidia-smi', '--query-gpu=memory.used,memory.free',
     '--format=csv,noheader,nounits'],
    capture_output=True, text=True)
print(f'VRAM: {r.stdout.strip()} (used, free MB)')
print(f'Model loaded on: {device}')
print('Model ready.')


## Cell 3 — Prompt Builders

Four prompt variants for each variable (Y and Z):

- **direct**: plain assignment — activates the Y attractor
- **ontological**: semantic category assertion — the patent intervention
- **direct_Z**: same as direct but with Z — control for token-specificity
- **ontological_Z**: ontological declaration with Z — control

The key comparison is **direct_Y vs direct_Z**: same prompt structure, different token. Any difference in trajectory geometry is attributable solely to the token's attractor properties.


In [ ]:
def prompt_direct_Y(y):
    """Standard assignment — known to activate the Y attractor."""
    return f'[INST] Y = {y}. What is Y + 1? [/INST]'

def prompt_ontological_Y(y):
    """Semantic category assertion — the patent intervention (Claim 4)."""
    return (
        f'[INST] Y is an algebraic variable number. '
        f'The numerical value of the variable Y is {y}. '
        f'What is Y + 1? [/INST]'
    )

def prompt_direct_Z(z):
    """Same structure as direct_Y but using Z — attractor-free control."""
    return f'[INST] Z = {z}. What is Z + 1? [/INST]'

def prompt_ontological_Z(z):
    """Ontological declaration with Z — control condition."""
    return (
        f'[INST] Z is an algebraic variable number. '
        f'The numerical value of the variable Z is {z}. '
        f'What is Z + 1? [/INST]'
    )

PROMPT_BUILDERS = {
    'direct_Y':       prompt_direct_Y,
    'ontological_Y':  prompt_ontological_Y,
    'direct_Z':       prompt_direct_Z,
    'ontological_Z':  prompt_ontological_Z,
}

# Show example prompts
for name, fn in PROMPT_BUILDERS.items():
    print(f'--- {name} (value=5) ---')
    print(fn(5))
    print()


## Cell 4 — Hidden State Trajectory Extraction

Registers forward hooks on each transformer layer to capture the mean-pooled hidden state at each layer during a single forward pass. This gives a trajectory: a sequence of 4096-dimensional vectors, one per layer, representing how the model's internal representation evolves from input to output.


In [ ]:
def extract_trajectory(prompt, layers=ALL_LAYERS):
    """
    Extract hidden states at each specified layer for a given prompt.
    Returns: dict {layer_idx: hidden_state_vector (4096,)}
    The hidden state is mean-pooled across the token dimension.
    """
    hidden_states = {}
    hooks = []

    def make_hook(layer_idx):
        def hook(module, input, output):
            # output[0] is the hidden state tensor: (batch, seq_len, hidden)
            h = output[0] if isinstance(output, tuple) else output
            # Mean pool across sequence length, detach and move to CPU
            hidden_states[layer_idx] = h.mean(dim=1).squeeze(0).detach().cpu().float()
        return hook

    # Register hooks on specified layers
    for layer_idx in layers:
        layer = model.model.layers[layer_idx]
        hooks.append(layer.register_forward_hook(make_hook(layer_idx)))

    try:
        inputs = tokenizer(prompt, return_tensors='pt',
                           truncation=True, max_length=256).to(device)
        with torch.no_grad():
            model(**inputs)
    finally:
        for h in hooks:
            h.remove()

    return hidden_states

# Test extraction
test_traj = extract_trajectory(prompt_direct_Y(5), layers=SAMPLE_LAYERS)
print('Trajectory extracted successfully')
print(f'Layers captured: {sorted(test_traj.keys())}')
print(f'Hidden state shape per layer: {test_traj[0].shape}')
print(f'Hidden dimension: {test_traj[0].shape[0]}')


## Cell 5 — Walsh-Hadamard Transform

The Walsh-Hadamard transform (WHT) distributes information across all coefficients equally — any localised structure in the original vector becomes spread across the transform domain. Crucially, it is an orthogonal transform: it preserves inner products and therefore preserves the geometric relationships (distances, angles, similarity) that encode the attractor structure.

We pad the 4096-dimensional hidden state to the next power of two if needed (4096 = 2^12, so no padding required for Mistral-7B) and apply the normalised WHT.

If the attractor is a genuine geometric feature, it should manifest as a **stable, sparse, consistent coefficient pattern** in the transform domain across different input values of Y — because the attractor basin corresponds to a fixed region in the representation space, and the WHT preserves that geometry.


In [ ]:
def next_power_of_two(n):
    p = 1
    while p < n:
        p <<= 1
    return p

def walsh_hadamard_transform(v):
    """
    Apply the normalised Walsh-Hadamard transform to vector v.
    v: numpy array of length 2^k (or will be zero-padded)
    Returns: WHT coefficients, normalised by 1/sqrt(N)
    """
    v = np.array(v, dtype=np.float32)
    n = len(v)
    p = next_power_of_two(n)
    if p != n:
        v = np.pad(v, (0, p - n))
    n = p

    # Fast Walsh-Hadamard transform (in-place butterfly)
    h = 1
    while h < n:
        for i in range(0, n, h * 2):
            for j in range(i, i + h):
                x = v[j]
                y = v[j + h]
                v[j]     = x + y
                v[j + h] = x - y
        h *= 2

    return v / np.sqrt(n)

def transform_trajectory(traj_dict, layers=None):
    """
    Apply WHT to each hidden state in a trajectory.
    Returns dict {layer_idx: wht_coefficients}
    """
    if layers is None:
        layers = sorted(traj_dict.keys())
    return {
        layer: walsh_hadamard_transform(traj_dict[layer].numpy())
        for layer in layers if layer in traj_dict
    }

# Validate: WHT should preserve vector norm (Parseval's theorem)
v_test = test_traj[0].numpy()
v_wht  = walsh_hadamard_transform(v_test)
norm_orig = np.linalg.norm(v_test)
norm_wht  = np.linalg.norm(v_wht)
print(f'Original norm:   {norm_orig:.4f}')
print(f'WHT norm:        {norm_wht:.4f}')
print(f'Norm preserved:  {np.isclose(norm_orig, norm_wht, rtol=1e-3)}')
print(f'WHT dimension:   {len(v_wht)}')


## Cell 6 — Collect All Trajectories

Runs all four prompt conditions across all Y values, extracting both raw and WHT-transformed trajectories at each sampled layer. This is the main data collection step — expect ~5-10 minutes on an RTX 3090.


In [ ]:
import time

print('Collecting trajectories...')
print(f'Conditions: {list(PROMPT_BUILDERS.keys())}')
print(f'Y values: {Y_VALUES}')
print(f'Layers: {SAMPLE_LAYERS}')
print(f'Total forward passes: {len(PROMPT_BUILDERS) * len(Y_VALUES)}')
print()

# Store: trajectories[condition][y_value] = {layer: hidden_state}
trajectories     = {cond: {} for cond in PROMPT_BUILDERS}
trajectories_wht = {cond: {} for cond in PROMPT_BUILDERS}

t0 = time.time()
for cond_name, prompt_fn in PROMPT_BUILDERS.items():
    print(f'Condition: {cond_name}')
    for y in Y_VALUES:
        prompt = prompt_fn(y)
        traj = extract_trajectory(prompt, layers=SAMPLE_LAYERS)
        trajectories[cond_name][y]     = traj
        trajectories_wht[cond_name][y] = transform_trajectory(traj)
        print(f'  y={y} done')
    print()

elapsed = time.time() - t0
print(f'All trajectories collected in {elapsed:.1f}s')

# Save raw data
save_data = {}
for cond in trajectories:
    save_data[cond] = {}
    for y in Y_VALUES:
        save_data[cond][str(y)] = {
            str(layer): trajectories[cond][y][layer].tolist()
            for layer in SAMPLE_LAYERS
        }

with open(DATA_DIR / 'wht_trajectories.json', 'w') as f:
    json.dump(save_data, f)
print('Trajectories saved to outputs/wht_trajectories.json')


## Experiment 1 — Input Sensitivity by Layer

**Key question**: do Y-attractor trajectories converge to the same geometric point regardless of input value, while Z-control trajectories remain separated?

Measure: pairwise cosine similarity between trajectories for different input values at each layer. If the attractor collapses all Y trajectories to the same basin, pairwise similarity will approach 1.0 at deeper layers regardless of input value. Z trajectories should maintain lower similarity, reflecting sensitivity to the input.


In [ ]:
from itertools import combinations

def mean_pairwise_cosine(traj_dict, y_values, layer):
    """
    Compute mean cosine similarity between all pairs of trajectories
    at a given layer. High similarity = trajectories have converged.
    """
    vecs = []
    for y in y_values:
        v = traj_dict[y][layer].numpy()
        vecs.append(v / (np.linalg.norm(v) + 1e-8))

    sims = []
    for v1, v2 in combinations(vecs, 2):
        sims.append(float(np.dot(v1, v2)))
    return np.mean(sims), np.std(sims)

print('Input sensitivity analysis')
print('Mean pairwise cosine similarity across Y values at each layer')
print('(Higher = trajectories have converged regardless of input value)')
print()

results_sensitivity = {}
for cond in ['direct_Y', 'direct_Z', 'ontological_Y', 'ontological_Z']:
    means, stds = [], []
    for layer in SAMPLE_LAYERS:
        m, s = mean_pairwise_cosine(trajectories[cond], Y_VALUES, layer)
        means.append(m)
        stds.append(s)
    results_sensitivity[cond] = {'means': means, 'stds': stds}

# Print table
header = f'{'Layer':>6}' + ''.join(f'{c:>18}' for c in ['direct_Y','direct_Z','onto_Y','onto_Z'])
print(header)
print('-' * len(header))
for i, layer in enumerate(SAMPLE_LAYERS):
    row = f'{layer:>6}'
    for cond in ['direct_Y', 'direct_Z', 'ontological_Y', 'ontological_Z']:
        m = results_sensitivity[cond]['means'][i]
        row += f'{m:>18.4f}'
    print(row)

print()
print('HYPOTHESIS: direct_Y should show higher similarity (convergence)')
print('than direct_Z at deeper layers — confirming attractor collapse.')


## Experiment 2 — WHT Coefficient Stability

**Key question**: does the Y attractor produce a stable, consistent coefficient pattern in the Walsh-Hadamard domain across different input values?

If the attractor corresponds to a fixed basin in representation space, then the WHT coefficients at deeper layers should be nearly identical across Y=3, Y=5, Y=7 etc. — because the WHT is order-preserving and the trajectories have all converged to the same attractor point.

For Z (no attractor), the WHT coefficients should differ across input values, reflecting genuine sensitivity to the input.


In [ ]:
print('WHT coefficient stability analysis')
print()

def wht_similarity(wht_traj, y_values, layer):
    """
    Mean pairwise cosine similarity of WHT-transformed hidden states.
    """
    vecs = []
    for y in y_values:
        v = wht_traj[y][layer]
        vecs.append(v / (np.linalg.norm(v) + 1e-8))
    sims = [float(np.dot(v1, v2))
            for v1, v2 in combinations(vecs, 2)]
    return np.mean(sims), np.std(sims)

results_wht = {}
for cond in ['direct_Y', 'direct_Z', 'ontological_Y', 'ontological_Z']:
    means, stds = [], []
    for layer in SAMPLE_LAYERS:
        m, s = wht_similarity(trajectories_wht[cond], Y_VALUES, layer)
        means.append(m)
        stds.append(s)
    results_wht[cond] = {'means': means, 'stds': stds}

print('WHT-domain pairwise cosine similarity (mean across Y value pairs):')
header = f'{'Layer':>6}' + ''.join(f'{c:>18}' for c in ['direct_Y','direct_Z','onto_Y','onto_Z'])
print(header)
print('-' * len(header))
for i, layer in enumerate(SAMPLE_LAYERS):
    row = f'{layer:>6}'
    for cond in ['direct_Y', 'direct_Z', 'ontological_Y', 'ontological_Z']:
        m = results_wht[cond]['means'][i]
        row += f'{m:>18.4f}'
    print(row)

print()
print('HYPOTHESIS: direct_Y WHT similarity should be high (basin convergence)')
print('direct_Z WHT similarity should be lower (input-sensitive)')
print('ontological_Y should be intermediate (intervention partially redirects)')


## Experiment 3 — Basin Geometry via PCA

Projects the WHT-transformed hidden states at the final layer into 2D using PCA, allowing direct visual inspection of the attractor basin geometry.

**Expected pattern for direct_Y**: all Y values cluster tightly in a single region — the attractor basin — regardless of input value.  
**Expected pattern for direct_Z**: points spread across the 2D space, separated by input value — no attractor collapse.  
**Expected pattern for ontological_Y**: points spread more than direct_Y, showing that the semantic category assertion has partially broken the attractor's geometric hold.


In [ ]:
print('PCA basin geometry analysis')
print()

# Use the deepest sampled layer (layer 31)
FINAL_LAYER = SAMPLE_LAYERS[-1]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle(
    f'WHT-Domain Attractor Basin Geometry (Layer {FINAL_LAYER})\n'
    'PCA projection of Walsh-Hadamard transformed hidden states\n'
    'Each point = one Y value; tight clustering = attractor collapse',
    fontsize=11
)

cond_titles = {
    'direct_Y':      'Direct Y (attractor active)',
    'direct_Z':      'Direct Z (no attractor — control)',
    'ontological_Y': 'Ontological Y (patent intervention)',
    'ontological_Z': 'Ontological Z (intervention control)',
}
cond_colors = {
    'direct_Y':      'red',
    'direct_Z':      'blue',
    'ontological_Y': 'orange',
    'ontological_Z': 'green',
}

pca_results = {}
for ax, (cond, title) in zip(axes.flat, cond_titles.items()):
    # Collect WHT vectors at final layer
    vecs = np.array([
        trajectories_wht[cond][y][FINAL_LAYER]
        for y in Y_VALUES
    ])

    # PCA to 2D
    pca = PCA(n_components=2)
    coords = pca.fit_transform(vecs)
    var_explained = pca.explained_variance_ratio_
    pca_results[cond] = {'coords': coords, 'variance': var_explained}

    # Plot
    scatter = ax.scatter(
        coords[:, 0], coords[:, 1],
        c=Y_VALUES, cmap='viridis', s=120,
        edgecolors=cond_colors[cond], linewidths=2, zorder=3
    )
    for i, y in enumerate(Y_VALUES):
        ax.annotate(f'Y={y}', (coords[i, 0], coords[i, 1]),
                    textcoords='offset points', xytext=(6, 4), fontsize=8)

    ax.set_title(f'{title}\nVar explained: PC1={var_explained[0]:.1%}, PC2={var_explained[1]:.1%}',
                 fontsize=9)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.grid(True, alpha=0.3)

    # Compute spread metric: mean pairwise Euclidean distance
    from itertools import combinations
    dists = [np.linalg.norm(coords[i] - coords[j])
             for i, j in combinations(range(len(Y_VALUES)), 2)]
    ax.set_xlabel(f'PC1 | Mean spread: {np.mean(dists):.3f}')

plt.tight_layout()
plt.savefig(DATA_DIR / 'wht_pca_basin_geometry.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to outputs/wht_pca_basin_geometry.png')


## Experiment 4 — Trajectory Convergence by Layer

Plots mean pairwise cosine similarity as a function of layer depth for all four conditions, in both the raw hidden state space and the WHT transform domain. This shows at which layer depth the attractor basin captures the trajectory and whether the WHT makes this transition more or less visible.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'direct_Y': 'red', 'direct_Z': 'blue',
          'ontological_Y': 'orange', 'ontological_Z': 'green'}
labels = {'direct_Y': 'Direct Y (attractor)', 'direct_Z': 'Direct Z (control)',
          'ontological_Y': 'Ontological Y (intervention)', 'ontological_Z': 'Ontological Z'}

# Left: raw hidden state similarity
ax = axes[0]
for cond in ['direct_Y', 'direct_Z', 'ontological_Y', 'ontological_Z']:
    means = results_sensitivity[cond]['means']
    stds  = results_sensitivity[cond]['stds']
    ax.plot(SAMPLE_LAYERS, means, 'o-', color=colors[cond],
            label=labels[cond], linewidth=2)
    ax.fill_between(SAMPLE_LAYERS,
                    [m-s for m,s in zip(means,stds)],
                    [m+s for m,s in zip(means,stds)],
                    alpha=0.15, color=colors[cond])

ax.set_title('Raw Hidden State Space\nMean pairwise cosine similarity across Y values',
             fontsize=10)
ax.set_xlabel('Layer depth')
ax.set_ylabel('Cosine similarity (higher = converged to same basin)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.5, 1.05)

# Right: WHT domain similarity
ax = axes[1]
for cond in ['direct_Y', 'direct_Z', 'ontological_Y', 'ontological_Z']:
    means = results_wht[cond]['means']
    stds  = results_wht[cond]['stds']
    ax.plot(SAMPLE_LAYERS, means, 's--', color=colors[cond],
            label=labels[cond], linewidth=2)
    ax.fill_between(SAMPLE_LAYERS,
                    [m-s for m,s in zip(means,stds)],
                    [m+s for m,s in zip(means,stds)],
                    alpha=0.15, color=colors[cond])

ax.set_title('Walsh-Hadamard Transform Domain\nMean pairwise cosine similarity across Y values',
             fontsize=10)
ax.set_xlabel('Layer depth')
ax.set_ylabel('WHT cosine similarity')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.5, 1.05)

plt.suptitle(
    'Attractor Basin Convergence: Raw vs Walsh-Hadamard Transform Domain\n'
    'Y attractor (red) should show higher similarity than Z control (blue)\n'
    'Ontological intervention (orange) should show intermediate values',
    fontsize=10
)
plt.tight_layout()
plt.savefig(DATA_DIR / 'wht_trajectory_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to outputs/wht_trajectory_convergence.png')


## Experiment 5 — WHT Coefficient Energy Distribution

Examines where the energy is concentrated in the WHT coefficient spectrum at each layer. A stable attractor should produce a consistent, sparse energy distribution — the same coefficients dominate regardless of input. An input-sensitive (non-attractor) representation should show variable energy distribution.

This connects directly to the DTDR work: the DTDR patent demonstrates that WHT preserves computational geometry under quantisation precisely because the energy distribution reflects the underlying semantic structure. If attractor states produce characteristic sparse WHT patterns, this supports both the DLMH hypothesis and the DTDR energy-preservation claim.


In [ ]:
print('WHT coefficient energy distribution analysis')
print()

# Use layer 31 (final)
ANALYSIS_LAYER = SAMPLE_LAYERS[-1]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(
    f'WHT Coefficient Energy Distribution at Layer {ANALYSIS_LAYER}\n'
    'Top-64 coefficients by magnitude | Each line = one Y value\n'
    'Stable attractor = overlapping lines; input-sensitive = separated lines',
    fontsize=10
)

TOP_K = 64  # Show top-K coefficients by mean magnitude

for ax, cond in zip(axes.flat, ['direct_Y','direct_Z','ontological_Y','ontological_Z']):
    # Collect WHT vectors for all Y values
    all_wht = np.array([
        trajectories_wht[cond][y][ANALYSIS_LAYER]
        for y in Y_VALUES
    ])  # shape: (n_y_values, hidden_dim)

    # Find top-K coefficients by mean absolute magnitude
    mean_mag = np.mean(np.abs(all_wht), axis=0)
    top_k_idx = np.argsort(mean_mag)[::-1][:TOP_K]
    top_k_idx_sorted = np.sort(top_k_idx)

    # Plot each Y value's coefficients at those positions
    cmap = plt.cm.viridis(np.linspace(0, 1, len(Y_VALUES)))
    for i, y in enumerate(Y_VALUES):
        coefs = all_wht[i][top_k_idx_sorted]
        ax.plot(range(TOP_K), coefs, alpha=0.7,
                color=cmap[i], linewidth=1.2, label=f'Y={y}')

    # Compute coefficient consistency: std across Y values at each position
    coef_std = np.std(all_wht[:, top_k_idx_sorted], axis=0)
    mean_std = np.mean(coef_std)

    title_map = {
        'direct_Y':      'Direct Y (attractor)',
        'direct_Z':      'Direct Z (control)',
        'ontological_Y': 'Ontological Y (intervention)',
        'ontological_Z': 'Ontological Z',
    }
    ax.set_title(f'{title_map[cond]}\nMean coefficient std: {mean_std:.4f} '
                 f'(lower = more consistent across Y values)', fontsize=8)
    ax.set_xlabel(f'Top-{TOP_K} coefficient index (by mean magnitude)')
    ax.set_ylabel('WHT coefficient value')
    ax.legend(fontsize=6, ncol=2)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(DATA_DIR / 'wht_coefficient_energy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to outputs/wht_coefficient_energy.png')


## Summary and Interpretation

This cell computes a summary statistics table and interprets the results in the context of the DLMH hypothesis.


In [ ]:
print('=' * 70)
print('WALSH-HADAMARD TRAJECTORY EXPERIMENT — SUMMARY')
print('=' * 70)
print()

FINAL_LAYER_IDX = len(SAMPLE_LAYERS) - 1

print('Mean pairwise cosine similarity at final layer (layer 31):')
print('(Measures whether trajectories for different Y values have converged)')
print()
print(f'  {"Condition":<22} {"Raw space":>12} {"WHT domain":>12} {"WHT/Raw ratio":>14}')
print(f'  {"-"*22} {"-"*12} {"-"*12} {"-"*14}')

for cond in ['direct_Y', 'direct_Z', 'ontological_Y', 'ontological_Z']:
    raw_sim = results_sensitivity[cond]['means'][FINAL_LAYER_IDX]
    wht_sim = results_wht[cond]['means'][FINAL_LAYER_IDX]
    ratio   = wht_sim / raw_sim if raw_sim > 0 else float('nan')
    print(f'  {cond:<22} {raw_sim:>12.4f} {wht_sim:>12.4f} {ratio:>14.4f}')

print()
print('Key comparisons:')
dy_raw = results_sensitivity['direct_Y']['means'][FINAL_LAYER_IDX]
dz_raw = results_sensitivity['direct_Z']['means'][FINAL_LAYER_IDX]
oy_raw = results_sensitivity['ontological_Y']['means'][FINAL_LAYER_IDX]

print(f'  Attractor effect (direct_Y vs direct_Z): {dy_raw - dz_raw:+.4f}')
print(f'  Intervention effect (onto_Y vs direct_Y): {oy_raw - dy_raw:+.4f}')
print()

print('DLMH interpretation:')
if dy_raw > dz_raw:
    print('  CONFIRMED: Y-attractor trajectories converge more strongly than')
    print('  Z-control trajectories, consistent with a genuine attractor basin.')
else:
    print('  UNEXPECTED: Y and Z show similar convergence.')
    print('  Consider: layer sampling, hidden state pooling method, or')
    print('  that the attractor may operate at the output rather than')
    print('  deep hidden state level.')

if oy_raw < dy_raw:
    print()
    print('  CONFIRMED: Ontological intervention reduces trajectory convergence,')
    print('  consistent with the semantic category assertion partially breaking')
    print('  the attractor basin capture.')

print()
print('Patent relevance:')
print('  These results provide geometric evidence that stable incorrect')
print('  convergence (Patent GB2607941.8) is a genuine attractor phenomenon')
print('  in the model representation space, not merely a surface output bias.')
print('  The Walsh-Hadamard transform preserves and reveals this geometry,')
print('  consistent with the DTDR framework (Patent GB2602157.6).')
print()
print('Citation:')
print('  West, J. (2026). Detecting and Modifying Stable Incorrect Convergence')
print('  in Probabilistic Inference Systems. UK Patent GB2607941.8.')
print('  West, J. (2026). Improving Memory Efficiency for Storing')
print('  High-Dimensional Numerical Data. UK Patent GB2602157.6.')
